Allows for single cell execution of Run_Retrievers notebook for multiple datasets

# Installation of dependencies 

In [1]:
!pip install ir_datasets


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip uninstall -y tensorflow tensorflow-gpu tensorflow-intel keras keras-nightly keras-preprocessing keras-tuner keras-cv keras-nlp tf-keras

In [4]:
!pip install "transformers==4.39.3" "sentence-transformers==2.7.0" torch "numpy<2.0"


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Imports and Config

In [3]:
from __future__ import annotations
import json, re
from typing import Dict, List, Tuple
from rank_bm25 import BM25Okapi
import ir_datasets
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import os

In [4]:
TOP_K = 100

def tokenize_simple(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "runs/"

In [13]:
DATASETS=["beir/cqadupstack/webmasters","beir/cqadupstack/unix","beir/cqadupstack/wordpress"] 

In [7]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF_WARNING"] = "1"

# Execution

In [14]:
for DATASET_NAME in DATASETS:
    dataset = ir_datasets.load(DATASET_NAME)
    print(f"Loaded {DATASET_NAME}")
    print(f"Docs: {dataset.docs_count()}, Queries: {dataset.queries_count()}")
    corpus_docs = list(dataset.docs_iter())
    doc_ids = [doc.doc_id for doc in corpus_docs]
    tokenized_docs = [tokenize_simple(doc.text) for doc in tqdm(corpus_docs, desc="Tokenizing corpus")]
    
    bm25 = BM25Okapi(tokenized_docs)
    print("BM25 index built successfully.")
    queries = list(dataset.queries_iter())
    qids = [q.query_id for q in queries]
    qtexts = [q.text for q in queries]
    
    run_records: List[Tuple[str, str, float]] = []
    
    for qid, qtext in tqdm(zip(qids, qtexts), total=len(qtexts), desc="Retrieving"):
        tokenized_query = tokenize_simple(qtext)
        scores = bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:TOP_K]
        for rank, idx in enumerate(top_indices, start=1):
            run_records.append((qid, doc_ids[idx], float(scores[idx])))
    
    print(f"Retrieved top-{TOP_K} docs for {len(qids)} queries.")
    OUT_PATH = f"runs/run_bm25_{DATASET_NAME.replace('/', '_')}.trec"
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for qid, docid, score in run_records:
            f.write(f"{qid} Q0 {docid} 0 {score:.6f} bm25\n")
    print(f"Run file saved at: {OUT_PATH}")

    #Dense computation below
    RUN_NAME = f"run_dense_{DATASET_NAME.replace('/', '_')}.trec"
    docs = list(dataset.docs_iter())
    doc_ids = [d.doc_id for d in docs]
    doc_texts = [d.text for d in docs]
    
    print(f"Encoding {len(docs)} documents with model {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    
    # Normalize embeddings → cosine similarity with FAISS inner product
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    dim = doc_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product (cosine if normalized)
    index.add(doc_embeddings)
    print(f"FAISS index built with {index.ntotal} documents.")
    
    queries = list(dataset.queries_iter())
    qids = [q.query_id for q in queries]
    qtexts = [q.text for q in queries]
    
    print(f"Encoding {len(queries)} queries...")
    query_embeddings = model.encode(
        qtexts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    print("Searching top-k similar documents...")
    D, I = index.search(query_embeddings, TOP_K)  # D = scores, I = indices
    run_path = os.path.join(OUTPUT_DIR, RUN_NAME)
    with open(run_path, "w", encoding="utf-8") as f:
        for qi, qid in enumerate(qids):
            pairs = [(doc_ids[idx], float(D[qi, j])) for j, idx in enumerate(I[qi])]
            pairs.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(pairs, start=1):
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} dense\n")
    
    print(f"✅ Dense run saved to: {run_path}")
    
    
    
    
    

    
    
    
    
    

[INFO] [starting] building docstore


Loaded beir/cqadupstack/webmasters
Docs: 17405, Queries: 506


[INFO] [starting] opening zip file                                              
[INFO] [finished] opening zip file s]                                        
docs_iter: 100%|██████████████████████| 17405/17405 [00:01<00:00, 13001.94doc/s]
[INFO] [finished] docs_iter: [00:01] [17405doc] [13001.94doc/s]
[INFO] [finished] building docstore [1.34s]
Tokenizing corpus: 100%|██████████████████████████████████████████████████████| 17405/17405 [00:00<00:00, 25177.01it/s]


BM25 index built successfully.


[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
Retrieving: 100%|████████████████████████████████████████████████████████████████████| 506/506 [00:58<00:00,  8.69it/s]


...

Retrieved top-100 docs for 541 queries.
Run file saved at: runs/run_bm25_beir_cqadupstack_wordpress.trec
Encoding 48605 documents with model sentence-transformers/all-MiniLM-L6-v2...


Batches:   0%|          | 0/190 [00:00<?, ?it/s]

FAISS index built with 48605 documents.
Encoding 541 queries...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Searching top-k similar documents...
✅ Dense run saved to: runs/run_dense_beir_cqadupstack_wordpress.trec
